# Introduction

This comprehensive notebook demonstrates **all capabilities** of **Gemini 3 Pro** - Google's most advanced multimodal AI model.

## What You'll Learn

### 🎯 API Fundamentals
- Single vs batch processing
- Streaming responses
- Context caching
- Error handling

### 📝 Text Modality (10 tasks)
1. Zero-shot sentiment analysis
2. Few-shot classification
3. Named Entity Recognition
4. Text summarization
5. Question answering
6. Translation
7. Text completion
8. Entity extraction
9. Keyword extraction
10. Text rewriting

### 🖼️ Vision Modality (8 tasks)
11. Object counting
12. Visual QA
13. Chart analysis
14. Document OCR
15. Image captioning
16. Multi-image comparison
17. Visual reasoning
18. Diagram understanding

### 🎬 Video Modality (3 tasks)
19. Video understanding
20. Frame-by-frame analysis
21. Action recognition

### 🎵 Audio Modality (2 tasks)
22. Speech transcription
23. Audio content analysis

### 📄 PDF Modality (2 tasks)
24. PDF document analysis
25. Multi-page extraction

### 🔧 Advanced Features (10 tasks)
26. Structured JSON output
27. Function calling
28. Code execution
29. Search grounding
30. Chain-of-thought (Thinking mode)
31. Long context (1M tokens)
32. Code generation
33. Mathematical reasoning
34. Scientific reasoning
35. Creative writing

## Model Specifications

**Model:** `gemini-2.0-flash-thinking-exp-1219`
- **Input modalities:** Text, Image, Video, Audio, PDF
- **Output:** Text only
- **Context:** 1,048,576 tokens input / 65,536 tokens output
- **Knowledge cutoff:** January 2025
- **Special features:** Thinking mode, Code execution, Context caching

# Setup and Configuration

In [5]:
# Install required packages
# !pip install google-genai pillow requests matplotlib pandas numpy

In [6]:
import os
import json
import time
from pathlib import Path
from typing import List, Dict, Any
from google import genai
from PIL import Image, ImageDraw, ImageFont
import requests
from io import BytesIO
import base64
import matplotlib.pyplot as plt
import numpy as np

# Check for API key
if 'GEMINI_API_KEY' not in os.environ:
    raise ValueError(
        "GEMINI_API_KEY not found in environment.\n"
        "Set it with: export GEMINI_API_KEY='your-key'\n"
        "Get your key at: https://aistudio.google.com/apikey"
    )

# Initialize client (new SDK)
client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])

print("✅ Gemini client initialized successfully")
print("Using google-genai SDK (new version)")

# Note: We'll use gemini-2.0-flash-thinking-exp-1219 as the default model
MODEL = "gemini-2.0-flash-thinking-exp-1219"
print(f"Default model: {MODEL}")

✅ Gemini client initialized successfully
Using google-genai SDK (new version)
Default model: gemini-2.0-flash-exp


## Helper Functions

In [7]:
def print_task(task_num: int, task_name: str):
    """Print formatted task header."""
    print("\n" + "="*80)
    print(f"TASK {task_num}: {task_name}")
    print("="*80)

def print_result(label: str, content: str, indent: int = 0):
    """Print formatted result."""
    prefix = "  " * indent
    print(f"{prefix}{label}: {content}")

def load_image_from_url(url: str) -> Image.Image:
    """Load an image from a URL."""
    response = requests.get(url)
    return Image.open(BytesIO(response.content))

def create_sample_image(text: str, size=(800, 600)) -> Image.Image:
    """Create a simple image with text for testing."""
    img = Image.new('RGB', size, color='white')
    draw = ImageDraw.Draw(img)
    draw.text((50, size[1]//2), text, fill='black')
    return img

print("Helper functions loaded")

Helper functions loaded


# Part 1: API Usage Patterns

## Single Request vs Batch Processing

In [10]:
print_task("0A", "Single Request")

# Single request - simplest way
response = client.models.generate_content(
    model="gemini-3-pro-preview",
    contents="What is the capital of France?")
print_result("Question", "What is the capital of France?")
print_result("Answer", response.text)


TASK 0A: Single Request
Question: What is the capital of France?
Answer: The capital of France is **Paris**.


In [ ]:
print_task("0B", "Batch Processing")

# Process multiple prompts efficiently
prompts = [
    "Translate 'Hello' to Spanish",
    "Translate 'Goodbye' to French",
    "Translate 'Thank you' to German",
    "Translate 'Welcome' to Italian"
]

# Method 1: Sequential (simple but slower)
print("Sequential processing:")
start = time.time()
results_seq = []
for prompt in prompts:
    response = client.models.generate_content(
    model="gemini-2.0-flash-thinking-exp-1219",
    contents=prompt)
    results_seq.append(response.text.strip())
time_seq = time.time() - start

for i, (prompt, result) in enumerate(zip(prompts, results_seq), 1):
    print(f"  {i}. {prompt} → {result}")
print(f"Time: {time_seq:.2f}s")


TASK 0B: Batch Processing
Sequential processing:
  1. Translate 'Hello' to Spanish → **Hola**
  2. Translate 'Goodbye' to French → The most common and direct translation of "Goodbye" in French is:

**Au revoir**

Here are a few other options depending on the context:

*   **Salut** (Informal, like "Bye" or "Hi" - used both for greeting and leaving)
*   **Adieu** (More definitive, like "farewell," often implying you won't see the person again for a very long time, if ever. Use with caution.)
*   **À bientôt** (See you soon)
*   **À tout à l'heure** (See you later today)
*   **À demain** (See you tomorrow)
*   **Bonne journée** (Have a good day)
*   **Bonne soirée** (Have a good evening)
*   **Bonne nuit** (Good night - only when parting ways for the night, usually before sleep)
  3. Translate 'Thank you' to German → The most common and direct translations for "Thank you" in German are:

1.  **Danke** (Thanks / Thank you - informal, or for smaller things)
2.  **Danke schön** (Thank you 

## Configuration and Safety Settings

In [17]:
print_task("0C", "Streaming Responses")

# Stream responses for long-running tasks
prompt = """Write a detailed explanation of how neural networks work,
covering architecture, training process, and applications."""

print("Streaming response (token by token):\n")

response = client.models.generate_content_stream(
    model="gemini-2.0-flash-thinking-exp-1219",
    contents=prompt
)

for chunk in response:
    if chunk.text:
        print(chunk.text, end='', flush=True)

print("\n\nStreaming complete!")


TASK 0D: Generation Configuration


TypeError: Models.generate_content() got an unexpected keyword argument 'generation_config'

# Part 2: Text-Only Tasks (10 tasks)

print_task("0D", "Generation Configuration")

from google.genai import types

# Configure generation parameters
generation_config = types.GenerateContentConfig(
    temperature=0.7,
    top_p=0.95,
    top_k=40,
    max_output_tokens=1024,
)

# Test with creative task
prompt = "Write a creative product name for an AI-powered coffee maker."

response = client.models.generate_content(
    model="gemini-2.0-flash-thinking-exp-1219",
    contents=prompt,
    config=generation_config
)

print(f"Configuration: temperature={generation_config.temperature}, top_p={generation_config.top_p}")
print(f"\nResult: {response.text}")

In [ ]:
print_task(1, "Zero-Shot Sentiment Analysis")

texts = [
    "This product is absolutely amazing! Best purchase I've made all year.",
    "Terrible experience. Waste of money and time.",
    "It's okay. Nothing special but does the job.",
    "I'm disappointed with the quality. Expected much better.",
    "Exceeded all my expectations! Highly recommend!"
]

prompt_template = """Classify the sentiment: Positive, Negative, or Neutral.
Reply with ONLY the sentiment label.

Text: {text}
Sentiment:"""

for i, text in enumerate(texts, 1):
    response = client.models.generate_content(
        model="gemini-2.0-flash-thinking-exp-1219",
        contents=prompt_template.format(text=text)
    )
    sentiment = response.text.strip()
    print(f"{i}. '{text[:50]}...'")
    print(f"   → {sentiment}\n")

## Task 2: Few-Shot Text Classification

In [ ]:
print_task(2, "Few-Shot Text Classification")

# Intent classification with examples
prompt = """Classify customer service queries into categories.

Examples:
"How do I reset my password?" → Technical Support
"I was charged twice" → Billing
"What are your hours?" → General Inquiry
"This is broken" → Complaint
"I want to cancel" → Account Management

Query: "{query}"
Category:"""

test_queries = [
    "My app keeps crashing when I upload photos",
    "Why was I charged for premium when I'm on free plan?",
    "Do you ship to Canada?",
    "The product arrived damaged",
    "How do I delete my account?"
]

for query in test_queries:
    response = client.models.generate_content(
        model="gemini-2.0-flash-thinking-exp-1219",
        contents=prompt.format(query=query)
    )
    print(f"Query: {query}")
    print(f"Category: {response.text.strip()}\n")

## Task 3: Named Entity Recognition

In [ ]:
print_task(3, "Named Entity Recognition (NER)")

text = """Apple Inc. CEO Tim Cook announced a $500 million investment in renewable 
energy projects across California next month. The announcement was made at the 
company's headquarters in Cupertino on December 15, 2024."""

prompt = f"""Extract all named entities and categorize them:
PERSON, ORGANIZATION, LOCATION, MONEY, DATE

Text: {text}

Format as JSON with entity type as key."""

response = client.models.generate_content(
    model="gemini-2.0-flash-thinking-exp-1219",
    contents=prompt)
print(f"Text: {text}\n")
print("Entities:")
print(response.text)

## Task 4: Text Summarization

In [ ]:
print_task(4, "Text Summarization")

article = """Artificial intelligence continues to transform industries worldwide. Recent
advances in large language models have enabled more natural conversations between humans
and machines. These models can understand context, generate coherent text, and even
perform complex reasoning tasks. However, challenges remain in ensuring factual accuracy,
reducing computational costs, and addressing ethical concerns around bias and privacy.
Researchers are actively working on making AI more efficient, transparent, and aligned
with human values. The field is evolving rapidly, with new breakthroughs announced weekly.
From healthcare to education, AI is reshaping how we work and live."""

prompts = [
    "Summarize in 1 sentence:",
    "Summarize in 3 bullet points:",
    "Create a tweet-length summary (280 chars):"
]

print(f"Original ({len(article)} chars):\n{article}\n")

for prompt_type in prompts:
    response = client.models.generate_content(
        model="gemini-2.0-flash-thinking-exp-1219",
        contents=f"{prompt_type}\n\n{article}"
    )
    print(f"{prompt_type}")
    print(f"  {response.text.strip()}\n")

## Task 5: Question Answering with Context

In [ ]:
print_task(5, "Question Answering")

context = """The Eiffel Tower is a wrought-iron lattice tower located on the Champ de Mars
in Paris, France. It was constructed from 1887 to 1889 as the centerpiece of the 1889
World's Fair. The tower is 330 meters (1,083 feet) tall, about the same height as an
81-story building. It was the tallest man-made structure in the world until the Chrysler
Building was completed in New York in 1930."""

questions = [
    "When was the Eiffel Tower built?",
    "How tall is the Eiffel Tower?",
    "Where is it located?",
    "What material is it made of?",
    "When did it stop being the tallest structure?"
]

print(f"Context: {context}\n")

for q in questions:
    prompt = f"Context: {context}\n\nQuestion: {q}\nAnswer (concise):"
    response = client.models.generate_content(
        model="gemini-2.0-flash-thinking-exp-1219",
        contents=prompt
    )
    print(f"Q: {q}")
    print(f"A: {response.text.strip()}\n")

## Task 6: Translation

In [ ]:
print_task(6, "Multi-Language Translation")

text = "Artificial intelligence is changing the world."
languages = ["Spanish", "French", "German", "Japanese", "Hindi", "Arabic"]

print(f"Original (English): {text}\n")

for lang in languages:
    prompt = f"Translate to {lang}: {text}"
    response = client.models.generate_content(
        model="gemini-2.0-flash-thinking-exp-1219",
        contents=prompt
    )
    print(f"{lang}: {response.text.strip()}")

## Task 7: Text Completion

In [ ]:
print_task(7, "Text Completion")

prompts = [
    "The secret to happiness is",
    "In the year 2050, technology will",
    "The most important skill for the future is"
]

for prompt in prompts:
    response = client.models.generate_content(
        model="gemini-2.0-flash-thinking-exp-1219",
        contents=f"Complete this sentence in 1-2 sentences: {prompt}"
    )
    print(f"Prompt: '{prompt}'")
    print(f"Completion: {response.text.strip()}\n")

## Task 8: Entity Extraction

In [ ]:
print_task(8, "Structured Entity Extraction")

resume = """JOHN DOE
john.doe@email.com | (555) 123-4567 | linkedin.com/in/johndoe

EXPERIENCE
Senior Software Engineer, TechCorp (2020-Present)
- Led team of 5 engineers in developing cloud infrastructure
- Expertise: Python, AWS, Docker, Kubernetes

EDUCATION
M.S. Computer Science, Stanford University (2018)
B.S. Computer Science, MIT (2016)"""

prompt = f"""Extract key information as JSON:
{{
  "name": "",
  "email": "",
  "phone": "",
  "current_role": "",
  "company": "",
  "skills": [],
  "education": []
}}

Resume:
{resume}

Return only valid JSON:"""

response = client.models.generate_content(
    model="gemini-2.0-flash-thinking-exp-1219",
    contents=prompt)
print("Extracted data:")
print(response.text)

## Task 9: Keyword Extraction

In [ ]:
print_task(9, "Keyword Extraction")

text = """Machine learning and deep learning are subsets of artificial intelligence 
that focus on training algorithms to recognize patterns in data. Neural networks, 
inspired by biological neurons, form the basis of deep learning systems. These 
technologies power applications like computer vision, natural language processing, 
and autonomous vehicles."""

prompt = f"""Extract the 5 most important keywords from this text.
Return as a comma-separated list.

Text: {text}

Keywords:"""

response = client.models.generate_content(
    model="gemini-2.0-flash-thinking-exp-1219",
    contents=prompt)
print(f"Text: {text}\n")
print(f"Keywords: {response.text.strip()}")

## Task 10: Text Rewriting

In [ ]:
print_task(10, "Text Rewriting for Different Audiences")

original = """The algorithm leverages advanced neural architectures to optimize
multi-dimensional parameter spaces through stochastic gradient descent."""

audiences = [
    "Explain to a 10-year-old",
    "Rewrite for a business executive",
    "Simplify for general audience",
    "Make it poetic"
]

print(f"Original: {original}\n")

for audience in audiences:
    response = client.models.generate_content(
        model="gemini-2.0-flash-thinking-exp-1219",
        contents=f"{audience}:\n\n{original}"
    )
    print(f"{audience}:")
    print(f"  {response.text.strip()}\n")

# Part 3: Vision Tasks (8 tasks)

## Task 11: Object Counting

In [ ]:
print_task(11, "Object Counting in Images")

# Create a test image with multiple objects
fig, ax = plt.subplots(figsize=(10, 8))
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis('off')

# Draw different shapes
circles = [(2, 2), (5, 5), (8, 3), (3, 7), (7, 8)]
squares_x = [1, 6, 9]
squares_y = [5, 2, 7]

for x, y in circles:
    circle = plt.Circle((x, y), 0.3, color='red', alpha=0.7)
    ax.add_patch(circle)

from matplotlib.patches import Rectangle
for x, y in zip(squares_x, squares_y):
    square = Rectangle((x-0.3, y-0.3), 0.6, 0.6, color='blue', alpha=0.7)
    ax.add_patch(square)

plt.title('Count the Objects', fontsize=16)
plt.savefig('/tmp/objects.png', dpi=150, bbox_inches='tight')
plt.close()

image = Image.open('/tmp/objects.png')

prompt = """Count the objects in this image:
1. How many red circles?
2. How many blue squares?
3. Total number of objects?"""

response = client.models.generate_content(
    model="gemini-2.0-flash-thinking-exp-1219",
    contents=[prompt, image])
print(response.text)

## Task 12: Visual Question Answering (VQA)

In [ ]:
print_task(12, "Visual Question Answering")

# Use sample images from URLs
try:
    image_url = "https://images.unsplash.com/photo-1506905925346-21bda4d32df4?w=800"
    image = load_image_from_url(image_url)

    questions = [
        "What is the dominant color in this image?",
        "Describe the scenery",
        "What time of day does it appear to be?",
        "What mood does this image convey?"
    ]

    for q in questions:
        response = client.models.generate_content(
            model="gemini-2.0-flash-thinking-exp-1219",
            contents=[q, image]
        )
        print(f"Q: {q}")
        print(f"A: {response.text.strip()}\n")

except Exception as e:
    print(f"Note: Image loading requires internet. Error: {str(e)[:100]}")

## Task 13: Chart and Graph Analysis

In [ ]:
print_task(13, "Chart Analysis")

# Create a complex chart
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Bar chart
months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun']
sales = [45000, 52000, 48000, 61000, 58000, 72000]
ax1.bar(months, sales, color='steelblue')
ax1.set_title('Monthly Sales 2024', fontsize=14, fontweight='bold')
ax1.set_ylabel('Sales ($)')
ax1.grid(axis='y', alpha=0.3)

# Line chart
days = list(range(1, 31))
visitors = [100 + 50*np.sin(x/5) + np.random.randint(-10, 10) for x in days]
ax2.plot(days, visitors, marker='o', linewidth=2, markersize=4)
ax2.set_title('Daily Website Visitors', fontsize=14, fontweight='bold')
ax2.set_xlabel('Day of Month')
ax2.set_ylabel('Visitors')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/charts.png', dpi=150)
plt.close()

chart_image = Image.open('/tmp/charts.png')

prompt = """Analyze these charts:
1. What trends do you see in the sales data?
2. Which month had the highest sales?
3. What pattern is visible in the website visitors chart?
4. Any notable insights?"""

response = client.models.generate_content(
    model="gemini-2.0-flash-thinking-exp-1219",
    contents=[prompt, chart_image])
print(response.text)

## Task 14: Document OCR and Understanding

In [ ]:
print_task(14, "Document OCR + Understanding")

# Create a sample receipt
img = Image.new('RGB', (600, 800), color='white')
draw = ImageDraw.Draw(img)

receipt_lines = [
    "ACME STORE",
    "123 Main Street",
    "Phone: (555) 123-4567",
    "",
    "Date: 2024-12-01",
    "Receipt #: 45678",
    "-" * 40,
    "Coffee Beans (2kg)      $24.99",
    "Milk (1L)                $3.49",
    "Bread                    $2.99",
    "Fresh Vegetables        $12.50",
    "-" * 40,
    "Subtotal:              $43.97",
    "Tax (8%):               $3.52",
    "TOTAL:                 $47.49",
    "",
    "Payment: VISA ****1234",
    "Thank you for shopping!"
]

y = 50
for line in receipt_lines:
    draw.text((50, y), line, fill='black')
    y += 35

img.save('/tmp/receipt.png')
receipt_img = Image.open('/tmp/receipt.png')

prompt = """Extract information from this receipt:
1. Store name and address
2. Date and receipt number
3. List of items purchased with prices
4. Total amount
5. Payment method

Format as structured JSON."""

response = client.models.generate_content(
    model="gemini-2.0-flash-thinking-exp-1219",
    contents=[prompt, receipt_img])
print(response.text)

## Task 15: Image Captioning

In [ ]:
print_task(15, "Image Captioning (Multiple Styles)")

try:
    image_url = "https://images.unsplash.com/photo-1506905925346-21bda4d32df4?w=800"
    image = load_image_from_url(image_url)

    caption_styles = [
        "Write a short caption (1 sentence)",
        "Write a detailed caption (2-3 sentences)",
        "Write an Instagram caption with hashtags",
        "Write a poetic caption"
    ]

    for style in caption_styles:
        response = client.models.generate_content(
            model="gemini-2.0-flash-thinking-exp-1219",
            contents=[style, image]
        )
        print(f"{style}:")
        print(f"  {response.text.strip()}\n")

except Exception as e:
    print(f"Using local image. Error: {str(e)[:100]}")

## Task 16: Multi-Image Comparison

In [ ]:
print_task(16, "Multi-Image Comparison")

# Create two different chart images
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Image 1: Pie chart
sizes = [30, 25, 20, 15, 10]
labels = ['A', 'B', 'C', 'D', 'E']
axes[0].pie(sizes, labels=labels, autopct='%1.1f%%')
axes[0].set_title('Product Distribution - Q1')

# Image 2: Pie chart with different values
sizes2 = [35, 20, 25, 10, 10]
axes[1].pie(sizes2, labels=labels, autopct='%1.1f%%')
axes[1].set_title('Product Distribution - Q2')

plt.tight_layout()
plt.savefig('/tmp/comparison.png', dpi=150)
plt.close()

comp_image = Image.open('/tmp/comparison.png')

prompt = """Compare these two pie charts:
1. What are the main differences?
2. Which products increased/decreased?
3. What insights can you derive?"""

response = client.models.generate_content(
    model="gemini-2.0-flash-thinking-exp-1219",
    contents=[prompt, comp_image])
print(response.text)

## Task 17: Visual Reasoning

In [ ]:
print_task(17, "Visual Pattern Reasoning")

# Create a visual pattern puzzle
fig, axes = plt.subplots(1, 4, figsize=(12, 3))

patterns = [
    {'shape': 'circle', 'color': 'red', 'size': 0.3},
    {'shape': 'square', 'color': 'blue', 'size': 0.4},
    {'shape': 'circle', 'color': 'red', 'size': 0.5},
    None  # To be predicted
]

for i, (ax, pattern) in enumerate(zip(axes, patterns)):
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')
    
    if pattern:
        if pattern['shape'] == 'circle':
            circle = plt.Circle((0.5, 0.5), pattern['size'], color=pattern['color'])
            ax.add_patch(circle)
        else:
            square = Rectangle(
                (0.5-pattern['size'], 0.5-pattern['size']),
                2*pattern['size'], 2*pattern['size'],
                color=pattern['color']
            )
            ax.add_patch(square)
    else:
        ax.text(0.5, 0.5, '?', fontsize=60, ha='center', va='center')
    
    ax.set_title(f'Position {i+1}')

plt.tight_layout()
plt.savefig('/tmp/pattern.png', dpi=150)
plt.close()

pattern_img = Image.open('/tmp/pattern.png')

prompt = """Analyze this visual pattern:
1. Describe the pattern you observe
2. What should appear in position 4?
3. Explain the logic"""

response = client.models.generate_content(
    model="gemini-2.0-flash-thinking-exp-1219",
    contents=[prompt, pattern_img])
print(response.text)

## Task 18: Diagram Understanding

In [ ]:
print_task(18, "Flow Diagram Understanding")

# Create a simple flowchart
fig, ax = plt.subplots(figsize=(10, 8))
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis('off')

# Draw boxes and arrows (simplified)
boxes = [
    {'text': 'Start', 'pos': (5, 9), 'color': 'lightgreen'},
    {'text': 'Input Data', 'pos': (5, 7), 'color': 'lightblue'},
    {'text': 'Process', 'pos': (5, 5), 'color': 'lightyellow'},
    {'text': 'Decision', 'pos': (5, 3), 'color': 'lightcoral'},
    {'text': 'Output', 'pos': (5, 1), 'color': 'lightgreen'},
]

for box in boxes:
    rect = Rectangle((box['pos'][0]-0.8, box['pos'][1]-0.4), 1.6, 0.8,
                     facecolor=box['color'], edgecolor='black', linewidth=2)
    ax.add_patch(rect)
    ax.text(box['pos'][0], box['pos'][1], box['text'],
           ha='center', va='center', fontsize=12, fontweight='bold')

# Add arrows
for i in range(len(boxes)-1):
    ax.arrow(boxes[i]['pos'][0], boxes[i]['pos'][1]-0.5,
            0, -1.2, head_width=0.3, head_length=0.2,
            fc='black', ec='black')

plt.title('Simple Flowchart', fontsize=16, fontweight='bold')
plt.savefig('/tmp/flowchart.png', dpi=150, bbox_inches='tight')
plt.close()

flowchart_img = Image.open('/tmp/flowchart.png')

prompt = """Analyze this flowchart:
1. Describe the process flow
2. How many steps are there?
3. What type of process does this represent?
4. Are there any decision points?"""

response = client.models.generate_content(
    model="gemini-2.0-flash-thinking-exp-1219",
    contents=[prompt, flowchart_img])
print(response.text)

# Part 4: Video Modality (3 tasks)

## Task 19: Video Understanding

In [ ]:
print_task(19, "Video Understanding")

print("""Note: Video analysis requires uploading video files to Gemini.
Example workflow:

```python
# Upload video file
video_file = genai.upload_file(path='path/to/video.mp4')

# Wait for processing
while video_file.state.name == 'PROCESSING':
    time.sleep(2)
    video_file = genai.get_file(video_file.name)

# Analyze video
prompt = 'Describe what happens in this video.'
response = client.models.generate_content(
    model="gemini-2.0-flash-thinking-exp-1219",
    contents=[prompt, video_file])
print(response.text)
```

Capabilities:
- Describe scenes and actions
- Count objects across frames
- Detect events and transitions
- Understand temporal relationships
- Answer questions about video content
""")

## Task 20: Frame-by-Frame Analysis

In [ ]:
print_task(20, "Frame-by-Frame Video Analysis")

print("""Example queries for frame-by-frame analysis:

1. Temporal Analysis:
   "At what timestamp does the person enter the scene?"
   "How long does the action last?"

2. Object Tracking:
   "Track the movement of the red car throughout the video"
   "Count how many times the ball bounces"

3. Scene Changes:
   "List all scene transitions with timestamps"
   "Describe the lighting changes"

4. Activity Recognition:
   "What activities are shown in chronological order?"
   "Identify all the actions performed by the main subject"
""")

## Task 21: Action Recognition

In [ ]:
print_task(21, "Action Recognition in Videos")

print("""Example prompts for action recognition:

```python
prompts = [
    # Simple action detection
    "Is someone walking, running, or standing still in this video?",
    
    # Multiple actions
    "List all distinct actions performed in chronological order",
    
    # Complex activities
    "Describe the cooking process shown in the video step by step",
    
    # Anomaly detection
    "Are there any unusual or unexpected actions in this video?",
    
    # Interaction analysis
    "Describe how the people in the video interact with each other"
]
```

Supported video formats: MP4, MOV, AVI, WebM
Max file size: 2GB
""")

# Part 5: Audio Modality (2 tasks)

## Task 22: Speech Transcription

In [ ]:
print_task(22, "Audio Transcription")

print("""Audio transcription workflow:

```python
# Upload audio file
audio_file = genai.upload_file(path='path/to/audio.mp3')

# Wait for processing
while audio_file.state.name == 'PROCESSING':
    time.sleep(1)
    audio_file = genai.get_file(audio_file.name)

# Transcribe
response = client.models.generate_content(
    model="gemini-2.0-flash-thinking-exp-1219",
    contents=[
    'Transcribe this audio exactly as spoken.',
    audio_file
])
print(response.text)
```

Capabilities:
- Transcribe speech to text
- Support for multiple languages
- Handle background noise
- Identify multiple speakers
- Add punctuation automatically

Supported formats: MP3, WAV, FLAC, OGG
""")

## Task 23: Audio Content Analysis

In [ ]:
print_task(23, "Audio Content Analysis")

print("""Beyond transcription, analyze audio content:

```python
analysis_prompts = [
    # Sentiment
    "What is the speaker's tone and emotion?",
    
    # Summarization
    "Summarize the main points discussed in this audio",
    
    # Speaker diarization
    "How many different speakers are in this audio? Describe each.",
    
    # Topic extraction
    "What topics are discussed in this audio file?",
    
    # Sound detection
    "What background sounds or music can you detect?",
    
    # Accent/language
    "What language is being spoken? Any notable accents?"
]

for prompt in analysis_prompts:
    response = client.models.generate_content(
    model="gemini-2.0-flash-thinking-exp-1219",
    contents=[prompt, audio_file])
    print(f"{prompt}\n{response.text}\n")
```
""")

# Part 6: PDF Modality (2 tasks)

## Task 24: PDF Document Analysis

In [ ]:
print_task(24, "PDF Document Understanding")

print("""PDF analysis workflow:

```python
# Upload PDF
pdf_file = genai.upload_file(path='path/to/document.pdf')

# Wait for processing
while pdf_file.state.name == 'PROCESSING':
    time.sleep(2)
    pdf_file = genai.get_file(pdf_file.name)

# Analyze document
prompts = [
    "Summarize this document",
    "Extract all tables and convert to JSON",
    "List all figures with their captions",
    "What are the main sections?",
    "Extract all numerical data",
    "Find all citations and references"
]

for prompt in prompts:
    response = client.models.generate_content(
    model="gemini-2.0-flash-thinking-exp-1219",
    contents=[prompt, pdf_file])
    print(f"{prompt}:\n{response.text}\n")
```

Capabilities:
- Extract text from any page
- Understand document structure
- Parse tables and convert to structured data
- Handle images within PDFs
- Maintain context across pages
""")

## Task 25: Multi-Page PDF Extraction

In [ ]:
print_task(25, "Multi-Page PDF Data Extraction")

print("""Example: Extract structured data from research papers

```python
pdf_file = genai.upload_file('research_paper.pdf')

extraction_prompt = '''Extract the following from this research paper:
{
  "title": "",
  "authors": [],
  "abstract": "",
  "keywords": [],
  "introduction": {
    "page": 0,
    "summary": ""
  },
  "methodology": {
    "page": 0,
    "summary": ""
  },
  "results": {
    "page": 0,
    "key_findings": []
  },
  "conclusions": "",
  "references_count": 0,
  "figures_count": 0,
  "tables_count": 0
}

Return valid JSON only.'''

response = client.models.generate_content(
    model="gemini-2.0-flash-thinking-exp-1219",
    contents=[extraction_prompt, pdf_file])
data = json.loads(response.text)
print(json.dumps(data, indent=2))
```

Use cases:
- Academic paper analysis
- Contract review
- Invoice processing
- Report summarization
- Form data extraction
""")

# Part 7: Advanced Features (10 tasks)

## Task 26: Structured JSON Output

In [ ]:
print_task(26, "Structured JSON Output")

text = """Sarah Johnson, 34, is a Senior Data Scientist at TechCorp in San Francisco. 
She specializes in machine learning and has 8 years of experience. Her skills include 
Python, TensorFlow, and SQL. Contact: sarah.j@techcorp.com, (555) 987-6543."""

schema = {
    "name": " ",
    "age": 0,
    "title": "",
    "company": "",
    "location": "",
    "experience_years": 0,
    "skills": [],
    "contact": {
        "email": "",
        "phone": ""
    }
}

prompt = f"""Extract information and return valid JSON matching this schema:
{json.dumps(schema, indent=2)}

Text: {text}

Return ONLY the JSON object, no markdown formatting:"""

response = client.models.generate_content(
    model="gemini-2.0-flash-thinking-exp-1219",
    contents=prompt)
result = response.text.strip()

# Clean up markdown if present
if result.startswith('```'):
    result = '\n'.join(result.split('\n')[1:-1])
    if result.startswith('json'):
        result = result[4:]

try:
    parsed = json.loads(result.strip())
    print("Extracted JSON:")
    print(json.dumps(parsed, indent=2))
except json.JSONDecodeError as e:
    print(f"Raw output:\n{result}")
    print(f"\nJSON parse error: {e}")

## Task 27: Function Calling

In [ ]:
print_task(27, "Function Calling / Tool Use")

print("""Note: Function calling in the new google-genai SDK uses a different approach.

Example workflow:

```python
from google.genai import types

# Define function schema
get_weather_func = types.FunctionDeclaration(
    name="get_current_weather",
    description="Get the current weather in a location",
    parameters={
        "type": "object",
        "properties": {
            "location": {
                "type": "string",
                "description": "City name"
            },
            "unit": {
                "type": "string",
                "enum": ["celsius", "fahrenheit"]
            }
        },
        "required": ["location"]
    }
)

# Create tool
weather_tool = types.Tool(function_declarations=[get_weather_func])

# Call with tools
response = client.models.generate_content(
    model="gemini-2.0-flash-thinking-exp-1219",
    contents="What's the weather in London?",
    config=types.GenerateContentConfig(
        tools=[weather_tool]
    )
)

# Check for function call
for part in response.candidates[0].content.parts:
    if hasattr(part, 'function_call'):
        func_call = part.function_call
        print(f"Function: {func_call.name}")
        print(f"Args: {dict(func_call.args)}")

        # Execute your function
        result = get_current_weather(**func_call.args)

        # Return result to model
        response2 = client.models.generate_content(
            model="gemini-2.0-flash-thinking-exp-1219",
            contents=[
                "What's the weather in London?",
                types.Content(parts=[types.Part.from_function_response(
                    name="get_current_weather",
                    response=result
                )])
            ]
        )
        print(response2.text)
```

Capabilities:
- Define custom functions for the model to call
- Handle structured data exchange
- Build agentic workflows
- Integrate with external APIs
""")

## Task 28: Code Execution

In [ ]:
print_task(28, "Code Execution")

print("""Code execution in the new google-genai SDK:

```python
from google.genai import types

# Enable code execution
code_execution_tool = types.Tool(code_execution={})

response = client.models.generate_content(
    model="gemini-2.0-flash-thinking-exp-1219",
    contents="Calculate the first 10 Fibonacci numbers",
    config=types.GenerateContentConfig(
        tools=[code_execution_tool]
    )
)

print(response.text)

# Check if code was executed
for part in response.candidates[0].content.parts:
    if hasattr(part, 'executable_code'):
        print(f"Code executed: {part.executable_code.code}")
    if hasattr(part, 'code_execution_result'):
        print(f"Result: {part.code_execution_result.output}")
```

Features:
- Execute Python code safely
- Perform calculations
- Generate visualizations
- Verify mathematical solutions
- Process data on-the-fly

Limitations:
- Sandboxed environment
- Limited libraries available
- No file system access
- No network access
""")

## Task 29: Search Grounding

In [ ]:
print_task(29, "Search Grounding (Google Search Integration)")

print("""Search grounding allows Gemini to search the web for up-to-date information:

```python
# Enable Google Search grounding
model_with_search = genai.GenerativeModel(
    'gemini-2.0-flash-thinking-exp-1219',
    tools='google_search_retrieval'
)

# Ask time-sensitive questions
queries = [
    "What are the latest developments in AI announced this week?",
    "What is the current stock price of NVIDIA?",
    "Who won the latest Nobel Prize in Physics?",
    "What are the top trending tech news today?"
]

for query in queries:
    response = model_with_search.generate_content(query)
    print(f"Q: {query}")
    print(f"A: {response.text}\n")
    
    # Access grounding metadata
    if hasattr(response, 'grounding_metadata'):
        print(f"Sources: {response.grounding_metadata}")
```

Benefits:
- Access to real-time information
- Factually grounded responses
- Citations and sources included
- Overcomes knowledge cutoff limitations
""")

## Task 30: Thinking Mode (Chain-of-Thought)

In [ ]:
print_task(30, "Thinking Mode - Extended Reasoning")

complex_problem = """A farmer has 17 sheep. All but 9 die. How many are left?
Explain your reasoning step by step."""

# Regular response
print("Without explicit thinking instructions:")
response_regular = client.models.generate_content(
    model="gemini-2.0-flash-thinking-exp-1219",
    contents=complex_problem
)
print(response_regular.text)

# With chain-of-thought prompting
print("\n" + "="*80)
print("With chain-of-thought reasoning:")

cot_prompt = f"""Let's solve this step by step:

{complex_problem}

Think through each step carefully:
Step 1: Identify what we know
Step 2: Identify what the question is asking
Step 3: Analyze the wording carefully
Step 4: Calculate the answer
Step 5: Verify the answer makes sense"""

response_cot = client.models.generate_content(
    model="gemini-2.0-flash-thinking-exp-1219",
    contents=cot_prompt
)
print(response_cot.text)

# More complex reasoning
print("\n" + "="*80)
print("Complex multi-step reasoning:")

logic_puzzle = """Three friends Alice, Bob, and Charlie have different jobs:
doctor, teacher, and engineer.
- The doctor is older than Alice
- Charlie is younger than the teacher
- Bob is not the youngest
- The engineer is the oldest

Who has which job? Explain your reasoning."""

response_puzzle = client.models.generate_content(
    model="gemini-2.0-flash-thinking-exp-1219",
    contents=f"Solve this logic puzzle step by step:\n\n{logic_puzzle}"
)
print(response_puzzle.text)

## Task 31: Long Context Understanding (1M Tokens)

In [ ]:
print_task(31, "Long Context Processing (1M Token Window)")

# Generate a substantial document
long_document = """
# Complete Machine Learning Course

## Module 1: Foundations

### Chapter 1: Introduction to Machine Learning
Machine learning is a method of data analysis that automates analytical model building.
It is a branch of artificial intelligence based on the idea that systems can learn from
data, identify patterns and make decisions with minimal human intervention.

Key concepts:
1. Supervised Learning: Learning from labeled examples
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through trial and error

### Chapter 2: Linear Regression
Linear regression is one of the simplest ML algorithms. It models the relationship
between a dependent variable and one or more independent variables.

Formula: y = mx + b
- y: predicted value
- m: slope
- x: input feature
- b: intercept

## Module 2: Deep Learning

### Chapter 3: Neural Networks
Neural networks are computing systems inspired by biological neural networks.
They consist of:
- Input layer
- Hidden layers
- Output layer

Activation functions:
1. ReLU: f(x) = max(0, x)
2. Sigmoid: f(x) = 1 / (1 + e^(-x))
3. Tanh: f(x) = (e^x - e^(-x)) / (e^x + e^(-x))

### Chapter 4: Convolutional Neural Networks
CNNs are specialized for processing grid-like data such as images.

Key components:
- Convolutional layers: Extract features
- Pooling layers: Reduce dimensionality
- Fully connected layers: Classification

## Module 3: Natural Language Processing

### Chapter 5: Word Embeddings
Word embeddings represent words as dense vectors in a continuous space.

Popular methods:
- Word2Vec (CBOW and Skip-gram)
- GloVe (Global Vectors)
- FastText

### Chapter 6: Transformers
Transformers use self-attention mechanisms to process sequential data.

Architecture:
- Multi-head attention
- Position-wise feed-forward networks
- Positional encoding

## Module 4: Practical Applications

### Chapter 7: Computer Vision
Applications:
- Image classification
- Object detection (YOLO, R-CNN)
- Semantic segmentation
- Face recognition

### Chapter 8: Recommendation Systems
Types:
1. Content-based filtering
2. Collaborative filtering
3. Hybrid approaches

## Summary
This course covered fundamental and advanced ML concepts, from basic linear
regression to state-of-the-art transformer architectures. Key takeaways include
understanding different learning paradigms, neural network architectures, and
practical applications in computer vision and NLP.
"""

# Demonstrate context retention
questions = [
    "What are the three types of machine learning mentioned in Chapter 1?",
    "What is the formula for linear regression from Chapter 2?",
    "List the activation functions mentioned in Chapter 3.",
    "What are the key components of CNNs from Chapter 4?",
    "Name the word embedding methods from Chapter 5.",
    "What are the main parts of transformer architecture mentioned in Chapter 6?",
    "List the computer vision applications from Chapter 7.",
    "What types of recommendation systems are discussed in Chapter 8?",
    "Provide a comprehensive summary of all modules covered."
]

print(f"Document size: ~{len(long_document.split())} words")
print(f"\nTesting context retention across the entire document:\n")

for i, q in enumerate(questions[:5], 1):  # Test first 5 for demo
    prompt = f"Based on this document:\n\n{long_document}\n\nQuestion: {q}\nAnswer:"
    response = client.models.generate_content(
        model="gemini-2.0-flash-thinking-exp-1219",
        contents=prompt
    )
    print(f"{i}. {q}")
    print(f"   {response.text.strip()}\n")

print("\n" + "="*80)
print("Note: Gemini 3 Pro can handle documents up to 1,048,576 tokens")
print("That's approximately ~750,000 words or ~1,500 pages!")

## Task 32: Code Generation

In [ ]:
print_task(32, "Code Generation")

code_tasks = [
    {
        "language": "Python",
        "task": """Create a decorator that measures function execution time
        and logs it with the function name."""
    },
    {
        "language": "JavaScript",
        "task": """Create an async function that fetches data from multiple
        URLs in parallel and returns combined results."""
    },
    {
        "language": "SQL",
        "task": """Write a query to find the top 5 customers by total
        purchase amount in the last 30 days."""
    }
]

for task in code_tasks:
    prompt = f"""Write {task['language']} code for this task:

{task['task']}

Include:
- Clean, production-ready code
- Type hints/comments where appropriate
- Error handling
- A brief explanation
"""

    response = client.models.generate_content(
        model="gemini-2.0-flash-thinking-exp-1219",
        contents=prompt
    )
    print(f"\n{task['language']} Task: {task['task'][:50]}...")
    print("-" * 80)
    print(response.text)
    print()

## Task 33: Mathematical Reasoning

In [ ]:
print_task(33, "Mathematical Problem Solving")

math_problems = [
    {
        "type": "Calculus",
        "problem": "Find the derivative of f(x) = (3x² + 2x - 1) * e^x"
    },
    {
        "type": "Linear Algebra",
        "problem": """Find the eigenvalues of the matrix:
        [[2, 1],
         [1, 2]]"""
    },
    {
        "type": "Statistics",
        "problem": """Given data: [12, 15, 18, 22, 25, 30, 35]
        Calculate: mean, median, variance, and standard deviation"""
    },
    {
        "type": "Optimization",
        "problem": """A rectangular garden has perimeter of 60m.
        What dimensions maximize the area?"""
    }
]

for prob in math_problems:
    prompt = f"""Solve this {prob['type']} problem step by step:

{prob['problem']}

Show all work and explain each step."""

    response = client.models.generate_content(
        model="gemini-2.0-flash-thinking-exp-1219",
        contents=prompt
    )
    print(f"\n{prob['type']}: {prob['problem'][:50]}...")
    print("-" * 80)
    print(response.text)
    print()

## Task 34: Scientific Reasoning

In [ ]:
print_task(34, "Scientific Reasoning")

scientific_questions = [
    {
        "field": "Physics",
        "question": """Explain the twin paradox in special relativity.
        Why does one twin age slower than the other?"""
    },
    {
        "field": "Chemistry",
        "question": """Why does entropy always increase in isolated systems?
        Explain with a molecular-level perspective."""
    },
    {
        "field": "Biology",
        "question": """How does CRISPR-Cas9 gene editing work?
        Explain the mechanism and its applications."""
    }
]

for sq in scientific_questions:
    prompt = f"""As a {sq['field']} expert, answer this question:

{sq['question']}

Provide a clear, accurate explanation suitable for an educated audience."""

    response = client.models.generate_content(
        model="gemini-2.0-flash-thinking-exp-1219",
        contents=prompt
    )
    print(f"\n{sq['field']}:")
    print("-" * 80)
    print(response.text)
    print()

## Task 35: Creative Writing

In [ ]:
print_task(35, "Creative Writing")

creative_tasks = [
    "Write a haiku about artificial intelligence",
    "Write a limerick about programming",
    "Write a short story (3 paragraphs) about time travel",
    "Write a product description for an AI-powered coffee maker",
    "Write a motivational speech for a data science team"
]

for i, task in enumerate(creative_tasks, 1):
    response = client.models.generate_content(
        model="gemini-2.0-flash-thinking-exp-1219",
        contents=task
    )
    print(f"\n{i}. {task}")
    print("-" * 80)
    print(response.text)
    print()

# Summary and Best Practices

## 🎯 What We Covered

This notebook demonstrated **35 tasks** across **all modalities** supported by Gemini 3 Pro:

### API Fundamentals
- Single vs batch processing
- Streaming responses
- Configuration and safety settings

### Text Modality (10 tasks)
✅ Sentiment analysis, classification, NER, summarization, QA, translation, completion, extraction, keywords, rewriting

### Vision Modality (8 tasks)
✅ Object counting, VQA, chart analysis, OCR, captioning, comparison, reasoning, diagrams

### Video Modality (3 tasks)
✅ Understanding, frame analysis, action recognition

### Audio Modality (2 tasks)
✅ Transcription, content analysis

### PDF Modality (2 tasks)
✅ Document analysis, multi-page extraction

### Advanced Features (10 tasks)
✅ Structured output, function calling, code execution, search grounding, thinking mode, long context, code generation, math, science, creative writing

## 📊 Performance Tips

1. **For Best Results:**
   - Be specific in prompts
   - Provide examples when possible
   - Use appropriate temperature settings
   - Structure outputs with schemas

2. **For Efficiency:**
   - Use context caching for repeated queries
   - Batch similar requests
   - Stream long responses
   - Optimize token usage

3. **For Accuracy:**
   - Enable search grounding for current events
   - Use code execution for calculations
   - Request step-by-step reasoning
   - Validate structured outputs

## 🔗 Resources

- [Gemini API Documentation](https://ai.google.dev/gemini-api/docs)
- [Gemini 3 Pro Model Card](https://ai.google.dev/gemini-api/docs/models#gemini-3-pro)
- [Google AI Studio](https://aistudio.google.com/)
- [Pricing](https://ai.google.dev/pricing)
- [Safety Settings](https://ai.google.dev/gemini-api/docs/safety-settings)

## 🚀 Next Steps

1. Test with your own use cases
2. Explore production features (caching, batching)
3. Build multimodal applications
4. Experiment with thinking mode for complex reasoning
5. Leverage the 1M token context for large documents

Happy building! 🎉